# Bagian A: Pra-pemrosesan Data
Membaca `data/data_sesudah_encoding.csv` (13 fitur + `pass_status`, nilai kosong sudah dibuang, ordinal sudah di-encode), lalu: cek data, split, outlier, oversampling, dan simpan CSV bersih untuk Gaussian Naïve Bayes (Bagian B).

## 0. Pengaturan dan pembacaan data

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# ---------- Pengaturan (ubah di sini bila perlu) ----------
PATH_ENCODED = "data/data_sesudah_encoding.csv"   # sumber
PATH_LATIH   = "data/data_latih_proses.csv"       # keluaran data latih
PATH_UJI     = "data/data_uji_proses.csv"         # keluaran data uji
TARGET       = "pass_status"
RANDOM_STATE = 42
UKURAN_UJI   = 0.20                               # 80:20

# Kolom numerik berskala interval dan rasio (11 kolom)
FITUR_NUMERIK = [
    "previous_exam_score", "previous_gpa", "attendance_percentage",
    "assignment_completion_rate", "study_hours_per_day", "self_study_hours",
    "practice_tests_completed", "exam_preparation_days", "sleep_hours",
    "stress_level", "exam_anxiety_level",
]
# Kolom ordinal (sudah berupa angka hasil encoding) dan rentang nilai yang sah
RENTANG_ORDINAL = {"exam_difficulty": (1, 3), "family_income": (1, 5)}
# Kolom yang di-winsorizing (miring ke kanan, nilai ekstrem masih logis)
KOLOM_WINSOR = ["study_hours_per_day", "self_study_hours", "practice_tests_completed"]

KOLOM_DIHARAPKAN = set(FITUR_NUMERIK) | set(RENTANG_ORDINAL) | {TARGET}

# ---------- Baca data ----------
data = pd.read_csv(PATH_ENCODED)
# Jika file menyimpan indeks asli sebagai kolom pertama, jadikan indeks kembali
for kol in ["index_asli", "Unnamed: 0"]:
    if kol in data.columns:
        data = data.set_index(kol)
        data.index.name = "index_asli"
        break

assert set(data.columns) == KOLOM_DIHARAPKAN, "Kolom tidak sesuai dengan 13 fitur + target"
print("Bentuk data:", data.shape)
data.head()

Bentuk data: (100000, 14)


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
0,78.36,2.95,81.49,65.18,4.94,2.63,8,27,6.31,10,9.39,2,3,Pass
1,73.40,2.89,NaN,76.53,3.42,2.64,10,4,6.81,8,6.31,1,3,Pass
2,82.76,3.35,100.00,70.39,1.06,0.83,5,28,6.53,9,9.87,1,3,Pass
3,60.61,2.57,84.91,75.06,7.89,5.51,8,16,6.60,10,9.68,2,3,Pass
4,74.79,2.84,88.21,61.54,2.63,2.29,1,28,7.83,10,10.00,2,5,Pass


## A1. Pengecekan duplikat dan inkonsistensi

In [2]:
print("Jumlah baris duplikat      :", data.duplicated().sum())
print("Jumlah nilai kosong (total) :", data.isna().sum().sum())

print("\nTipe data setiap kolom:")
print(data.dtypes.astype(str).to_string())

print("\nNilai unik kolom ordinal (hasil encoding):")
for kol, (lo, hi) in RENTANG_ORDINAL.items():
    nilai = sorted(data[kol].dropna().unique().tolist())
    print(f"  {kol}: {nilai}")
    assert data[kol].dropna().between(lo, hi).all(), f"{kol} di luar rentang {lo}-{hi}"

print("\nNilai unik target:", sorted(data[TARGET].unique()))

Jumlah baris duplikat      : 0
Jumlah nilai kosong (total) : 17656

Tipe data setiap kolom:
previous_exam_score           float64
previous_gpa                  float64
attendance_percentage         float64
assignment_completion_rate    float64
study_hours_per_day           float64
self_study_hours              float64
practice_tests_completed        int64
exam_preparation_days           int64
sleep_hours                   float64
stress_level                    int64
exam_anxiety_level            float64
exam_difficulty                 int64
family_income                   int64
pass_status                    object

Nilai unik kolom ordinal (hasil encoding):
  exam_difficulty: [1, 2, 3]
  family_income: [1, 2, 3, 4, 5]

Nilai unik target: ['Fail', 'Pass']


## A2. Penanganan nilai kosong (buang baris yang kosong)

In [3]:
def distribusi(seri):
    jumlah = seri.value_counts()
    return pd.DataFrame({"Jumlah": jumlah, "Persen (%)": (jumlah / jumlah.sum() * 100).round(2)})

n_sebelum = len(data)
kosong_sebelum = data.isna().sum()
dist_sebelum = distribusi(data[TARGET])

data = data.dropna()          # indeks asli TIDAK di-reset

print(f"Jumlah baris sebelum : {n_sebelum}")
print(f"Jumlah baris sesudah : {len(data)}  (dibuang {n_sebelum - len(data)} baris, "
      f"{(n_sebelum - len(data)) / n_sebelum * 100:.2f}%)")
print("\nNilai kosong per kolom SEBELUM:")
print(kosong_sebelum[kosong_sebelum > 0].to_string() if kosong_sebelum.sum() else "  tidak ada")
print("Nilai kosong per kolom SESUDAH :", data.isna().sum().sum())
print("\nDistribusi pass_status SEBELUM:"); print(dist_sebelum)
print("\nDistribusi pass_status SESUDAH:"); print(distribusi(data[TARGET]))

Jumlah baris sebelum : 100000
Jumlah baris sesudah : 83125  (dibuang 16875 baris, 16.88%)

Nilai kosong per kolom SEBELUM:
previous_gpa             7793
attendance_percentage    9863
Nilai kosong per kolom SESUDAH : 0

Distribusi pass_status SEBELUM:
             Jumlah  Persen (%)
pass_status                    
Pass          77389       77.39
Fail          22611       22.61

Distribusi pass_status SESUDAH:
             Jumlah  Persen (%)
pass_status                    
Pass          64272       77.32
Fail          18853       22.68


## A3. Pembagian data (80:20, stratified)

In [4]:
data_latih, data_uji = train_test_split(
    data, test_size=UKURAN_UJI, stratify=data[TARGET], random_state=RANDOM_STATE
)
data_latih = data_latih.copy()
data_uji = data_uji.copy()

print(f"Data latih: {data_latih.shape}   Data uji: {data_uji.shape}")
print("\nDistribusi pass_status data latih:"); print(distribusi(data_latih[TARGET]))
print("\nDistribusi pass_status data uji:");   print(distribusi(data_uji[TARGET]))

Data latih: (66500, 14)   Data uji: (16625, 14)

Distribusi pass_status data latih:
             Jumlah  Persen (%)
pass_status                    
Pass          51418       77.32
Fail          15082       22.68

Distribusi pass_status data uji:
             Jumlah  Persen (%)
pass_status                    
Pass          12854       77.32
Fail           3771       22.68


## A4. Analisis outlier (batas IQR dihitung dari data latih saja)

In [5]:
def batas_iqr(seri):
    q1, q3 = seri.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

baris = []
for kol in FITUR_NUMERIK:
    s = data_latih[kol]
    bawah, atas = batas_iqr(s)
    n_bawah, n_atas = (s < bawah).sum(), (s > atas).sum()
    baris.append({
        "kolom": kol, "min": s.min(), "max": s.max(),
        "batas_bawah": round(bawah, 2), "batas_atas": round(atas, 2),
        "n_bawah": n_bawah, "n_atas": n_atas,
        "persen_outlier": round((n_bawah + n_atas) / len(s) * 100, 2),
        "skewness": round(s.skew(), 2),
    })
tabel_outlier = pd.DataFrame(baris).set_index("kolom")
tabel_outlier

,min,max,batas_bawah,batas_atas,n_bawah,n_atas,persen_outlier,skewness
kolom,,,,,,,,
previous_exam_score,30.00,100.0,38.62,100.50,213,0,0.32,-0.01
previous_gpa,1.00,4.0,1.49,4.08,211,0,0.32,-0.02
attendance_percentage,46.34,100.0,62.78,106.59,256,0,0.38,-0.17
assignment_completion_rate,20.00,100.0,36.86,102.54,205,0,0.31,0.02
study_hours_per_day,0.50,12.0,-1.57,7.20,0,1742,2.62,1.13
self_study_hours,0.25,10.0,-1.17,5.07,0,1963,2.95,1.28
practice_tests_completed,0.00,20.0,-0.50,11.50,0,1175,1.77,0.53
exam_preparation_days,1.00,30.0,-14.50,45.50,0,0,0.00,0.00
sleep_hours,3.00,11.0,3.73,10.25,230,215,0.67,-0.00


## A5. Winsorizing (batas dari data latih, diterapkan ke data latih dan uji)

In [6]:
batas_winsor = {}
for kol in KOLOM_WINSOR:
    bawah, atas = batas_iqr(data_latih[kol])
    if pd.api.types.is_integer_dtype(data_latih[kol]):    # kolom cacah: batas dibuat bilangan bulat
        bawah, atas = np.ceil(bawah), np.floor(atas)
    batas_winsor[kol] = (bawah, atas)

print("Batas winsorizing (dari data latih):")
for kol, (b, a) in batas_winsor.items():
    print(f"  {kol}: bawah={b:.2f}, atas={a:.2f}")

ringkasan = []
for nama, df_ in [("latih", data_latih), ("uji", data_uji)]:
    for kol, (b, a) in batas_winsor.items():
        asli = df_[kol].copy()
        df_[kol] = df_[kol].clip(lower=b, upper=a).astype(asli.dtype)
        ringkasan.append({"data": nama, "kolom": kol, "jumlah_nilai_berubah": int((asli != df_[kol]).sum()),
                          "max_sebelum": asli.max(), "max_sesudah": df_[kol].max()})
pd.DataFrame(ringkasan)

Batas winsorizing (dari data latih):
  study_hours_per_day: bawah=-1.57, atas=7.20
  self_study_hours: bawah=-1.17, atas=5.07
  practice_tests_completed: bawah=-0.00, atas=11.00


,data,kolom,jumlah_nilai_berubah,max_sebelum,max_sesudah
0,latih,study_hours_per_day,1742,12.0,7.195
1,latih,self_study_hours,1963,10.0,5.070
2,latih,practice_tests_completed,1175,20.0,11.000
3,uji,study_hours_per_day,406,12.0,7.195
4,uji,self_study_hours,499,10.0,5.070
5,uji,practice_tests_completed,276,19.0,11.000


## A6. Transformasi kategorikal ke numerik (ordinal)
Ordinal encoding sudah dilakukan pada `data_sesudah_encoding.csv`. Di sini hanya diverifikasi.

- `exam_difficulty`: Easy = 1, Medium = 2, Hard = 3
- `family_income`: Low = 1, Lower-Middle = 2, Middle = 3, Upper-Middle = 4, High = 5

In [7]:
for nama, df_ in [("latih", data_latih), ("uji", data_uji)]:
    for kol, (lo, hi) in RENTANG_ORDINAL.items():
        assert df_[kol].between(lo, hi).all(), f"{kol} ({nama}) di luar rentang"
    assert df_.isna().sum().sum() == 0, f"Ada NaN pada data {nama}"
print("Verifikasi ordinal encoding lolos: tidak ada NaN, semua nilai dalam rentang.")
for kol in RENTANG_ORDINAL:
    print(kol, "-> nilai pada data latih:", sorted(data_latih[kol].unique().tolist()))

Verifikasi ordinal encoding lolos: tidak ada NaN, semua nilai dalam rentang.
exam_difficulty -> nilai pada data latih: [1, 2, 3]
family_income -> nilai pada data latih: [1, 2, 3, 4, 5]


## A7. Penanganan data tidak seimbang (oversampling, data latih saja)

In [8]:
dist_latih_sebelum = distribusi(data_latih[TARGET])

jumlah = data_latih[TARGET].value_counts()
kelas_mayor, kelas_minor = jumlah.idxmax(), jumlah.idxmin()
mayor = data_latih[data_latih[TARGET] == kelas_mayor]
minor = data_latih[data_latih[TARGET] == kelas_minor]

minor_naik = resample(minor, replace=True, n_samples=len(mayor), random_state=RANDOM_STATE)
data_latih_os = pd.concat([mayor, minor_naik]).sample(frac=1, random_state=RANDOM_STATE)  # acak urutan

print(f"Kelas mayoritas: {kelas_mayor}   Kelas minoritas: {kelas_minor}")
print("\nDistribusi data latih SEBELUM oversampling:"); print(dist_latih_sebelum)
print("\nDistribusi data latih SESUDAH oversampling:"); print(distribusi(data_latih_os[TARGET]))
print("\nData uji tidak diubah:"); print(distribusi(data_uji[TARGET]))

Kelas mayoritas: Pass   Kelas minoritas: Fail

Distribusi data latih SEBELUM oversampling:
             Jumlah  Persen (%)
pass_status                    
Pass          51418       77.32
Fail          15082       22.68

Distribusi data latih SESUDAH oversampling:
             Jumlah  Persen (%)
pass_status                    
Pass          51418        50.0
Fail          51418        50.0

Data uji tidak diubah:
             Jumlah  Persen (%)
pass_status                    
Pass          12854       77.32
Fail           3771       22.68


## A8. Hasil akhir pra-pemrosesan

In [9]:
X_train, y_train = data_latih_os.drop(columns=TARGET), data_latih_os[TARGET]
X_test,  y_test  = data_uji.drop(columns=TARGET),      data_uji[TARGET]

print("X_train:", X_train.shape, "  y_train:", y_train.shape)
print("X_test :", X_test.shape,  "  y_test :", y_test.shape)

print("\n5 baris teratas data latih hasil pemrosesan:")
display(data_latih_os.head(5))
print("5 baris terbawah data latih hasil pemrosesan:")
display(data_latih_os.tail(5))

X_train:

 (102836, 13)   y_train: (102836,)
X_test : (16625, 13)   y_test : (16625,)

5 baris teratas data latih hasil pemrosesan:


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
65155,76.54,3.21,84.21,74.61,1.80,1.22,8,13,6.13,10,10.00,2,1,Pass
71522,67.70,2.86,74.99,60.38,0.69,0.57,8,28,4.51,10,10.00,2,3,Fail
84297,55.58,2.28,91.08,65.58,2.95,1.80,7,11,6.74,9,9.38,2,3,Fail
5865,47.58,1.87,78.45,67.99,1.11,0.73,3,3,7.16,10,9.81,2,2,Fail
84902,84.76,3.34,79.27,71.37,4.33,3.85,5,25,7.70,8,8.86,2,5,Pass


5 baris terbawah data latih hasil pemrosesan:


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
73757,77.58,3.00,83.62,43.56,0.68,0.38,2,20,5.85,10,7.18,2,2,Pass
29650,55.95,2.39,75.07,52.74,1.19,0.91,4,23,8.03,10,10.00,1,5,Fail
66681,46.98,1.98,88.47,80.42,6.10,3.45,4,12,6.57,9,8.34,3,3,Fail
43910,72.14,3.02,77.23,63.65,0.81,0.73,2,19,8.43,8,9.54,1,3,Pass
66935,43.69,1.93,82.90,71.54,4.34,2.67,5,19,5.95,9,10.00,1,2,Pass


## A9. Simpan hasil ke CSV

In [10]:
os.makedirs(os.path.dirname(PATH_LATIH), exist_ok=True)
data_latih_os.to_csv(PATH_LATIH, index=True, index_label="index_asli")
data_uji.to_csv(PATH_UJI, index=True, index_label="index_asli")
print("Tersimpan:", PATH_LATIH, "dan", PATH_UJI)

Tersimpan: data/data_latih_proses.csv dan data/data_uji_proses.csv


## A10. Verifikasi file yang disimpan (baca ulang)

In [11]:
latih_b = pd.read_csv(PATH_LATIH, index_col="index_asli")
uji_b   = pd.read_csv(PATH_UJI,   index_col="index_asli")

assert latih_b.shape == data_latih_os.shape and uji_b.shape == data_uji.shape
assert latih_b.isna().sum().sum() == 0 and uji_b.isna().sum().sum() == 0
assert list(latih_b.columns) == list(uji_b.columns)
pd.testing.assert_frame_equal(latih_b, data_latih_os, check_dtype=False)
pd.testing.assert_frame_equal(uji_b, data_uji, check_dtype=False)
print("Verifikasi lolos: isi file sama dengan data di memori.")

print("\nShape latih:", latih_b.shape, " uji:", uji_b.shape)
print("\nDistribusi pass_status (latih):"); print(distribusi(latih_b[TARGET]))
print("\nDistribusi pass_status (uji):");   print(distribusi(uji_b[TARGET]))
print("\n5 baris teratas data latih (hasil baca ulang):");  display(latih_b.head(5))
print("5 baris terbawah data latih (hasil baca ulang):");    display(latih_b.tail(5))

Verifikasi lolos: isi file sama dengan data di memori.

Shape latih: (102836, 14)  uji: (16625, 14)

Distribusi pass_status (latih):
             Jumlah  Persen (%)
pass_status                    
Pass          51418        50.0
Fail          51418        50.0

Distribusi pass_status (uji):
             Jumlah  Persen (%)
pass_status                    
Pass          12854       77.32
Fail           3771       22.68

5 baris teratas data latih (hasil baca ulang):


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
65155,76.54,3.21,84.21,74.61,1.80,1.22,8,13,6.13,10,10.00,2,1,Pass
71522,67.70,2.86,74.99,60.38,0.69,0.57,8,28,4.51,10,10.00,2,3,Fail
84297,55.58,2.28,91.08,65.58,2.95,1.80,7,11,6.74,9,9.38,2,3,Fail
5865,47.58,1.87,78.45,67.99,1.11,0.73,3,3,7.16,10,9.81,2,2,Fail
84902,84.76,3.34,79.27,71.37,4.33,3.85,5,25,7.70,8,8.86,2,5,Pass


5 baris terbawah data latih (hasil baca ulang):


,previous_exam_score,previous_gpa,attendance_percentage,assignment_completion_rate,study_hours_per_day,self_study_hours,practice_tests_completed,exam_preparation_days,sleep_hours,stress_level,exam_anxiety_level,exam_difficulty,family_income,pass_status
index_asli,,,,,,,,,,,,,,
73757,77.58,3.00,83.62,43.56,0.68,0.38,2,20,5.85,10,7.18,2,2,Pass
29650,55.95,2.39,75.07,52.74,1.19,0.91,4,23,8.03,10,10.00,1,5,Fail
66681,46.98,1.98,88.47,80.42,6.10,3.45,4,12,6.57,9,8.34,3,3,Fail
43910,72.14,3.02,77.23,63.65,0.81,0.73,2,19,8.43,8,9.54,1,3,Pass
66935,43.69,1.93,82.90,71.54,4.34,2.67,5,19,5.95,9,10.00,1,2,Pass
